# QLoRA Fine-tuning: Llama 3.1 8B on Cybersecurity QA Dataset

**Project**: Cybersecurity Domain RAG Pipeline — Final Year Project (CB013309)  
**Technique**: QLoRA (Quantised Low-Rank Adaptation)  
**Base model**: `meta-llama/Llama-3.1-8B`

---

## Overview

This notebook fine-tunes Meta's Llama 3.1 8B on a curated cybersecurity QA dataset using **QLoRA** — a parameter-efficient fine-tuning technique that combines 4-bit quantisation with low-rank adapter matrices. The resulting adapter is designed to be plugged into the RAG pipeline as Step 5 (fine-tuned model), enabling side-by-side evaluation against the baseline in the Streamlit evaluation dashboard.

### Why QLoRA?

| Property | Full Fine-tuning | LoRA | **QLoRA** |
|---|---|---|---|
| VRAM required | ~40 GB (A100) | ~16 GB | **~4–6 GB (T4 free)** |
| Trainable parameters | 8 B (100%) | ~20 M (0.25%) | **~20 M (0.25%)** |
| Relative quality | 100% | ~95% | **~90%** |

The base model weights are **frozen** and stored in 4-bit NF4 precision. Only the small LoRA adapter matrices are trained, reducing GPU memory from ~16 GB to ~4–5 GB — fitting comfortably on a free Colab T4.

### Dataset

- **45 cybersecurity documents** covering SQL injection, XSS, ransomware, cryptography, zero-trust, GDPR/PCI DSS/ISO 27001, Active Directory, container security, and more  
- **225 QA pairs** generated using Llama 3.1 8B via Groq; answers grounded strictly in document content  
- Training format: `Question: {q}\nContext: {full_doc}\nAnswer: {a}`

### Prerequisites

1. **GPU runtime**: Runtime → Change runtime type → **T4 GPU**  
2. **HuggingFace account** with Llama 3.1 licence accepted at https://huggingface.co/meta-llama/Llama-3.1-8B  
3. **`cybersecurity_qa.jsonl`** uploaded to Google Drive (path configured in Cell 4)

---
## Step 1 — Install Dependencies

| Package | Purpose |
|---|---|
| `transformers` | Model loading, tokeniser, text generation |
| `peft` | LoRA configuration and adapter management |
| `trl` | `SFTTrainer` — supervised fine-tuning with PEFT support |
| `bitsandbytes` | 4-bit NF4 quantisation (Linux only — available on Colab) |
| `datasets` | HuggingFace dataset loading and formatting |
| `accelerate` | Distributed training / device placement backend |

In [ ]:
!pip install -q \
    transformers \
    peft \
    trl \
    bitsandbytes \
    datasets \
    accelerate \
    huggingface_hub

import torch

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---
## Step 2 — Mount Google Drive and Configure Paths

The cell below mounts your Drive and **automatically locates `cybersecurity_qa.jsonl`** wherever you placed it — no need to set a manual path. It searches your entire Drive and picks the first match.

The fine-tuned LoRA adapter weights will be saved to `My Drive/fyp_finetuned/lora_adapter/` so they persist after the Colab session ends.

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

# ── Configure these paths to match your Drive layout ──────────────────────
DRIVE_JSONL   = "/content/drive/MyDrive/fyp_data/cybersecurity_qa.jsonl"
DRIVE_OUTPUT  = "/content/drive/MyDrive/fyp_finetuned"
ADAPTER_DIR   = os.path.join(DRIVE_OUTPUT, "lora_adapter")
# ──────────────────────────────────────────────────────────────────────────

os.makedirs(ADAPTER_DIR, exist_ok=True)

assert os.path.exists(DRIVE_JSONL), (
    f"JSONL not found at {DRIVE_JSONL}. "
    "Upload training_data/cybersecurity_qa.jsonl to Drive first."
)

import subprocess
line_count = int(subprocess.check_output(["wc", "-l", DRIVE_JSONL]).split()[0])
print(f"Dataset  : {DRIVE_JSONL}")
print(f"Pairs    : {line_count}")
print(f"Adapter  : {ADAPTER_DIR}")

---
## Step 3 — HuggingFace Authentication

`meta-llama/Llama-3.1-8B` is a **gated model** — access requires:

1. A HuggingFace account (free): https://huggingface.co/join  
2. Accepting the Meta Llama 3.1 Community Licence at https://huggingface.co/meta-llama/Llama-3.1-8B  
3. A **read-access token** from https://huggingface.co/settings/tokens

Paste your token in the prompt below. It is not stored in notebook output.

In [ ]:
from huggingface_hub import login

login()   # enter your HF read token when prompted
print("Authenticated with HuggingFace.")

---
## Step 4 — Load Base Model with 4-bit Quantisation

### BitsAndBytesConfig parameters

| Parameter | Value | Reason |
|---|---|---|
| `load_in_4bit` | `True` | Quantise weights to 4 bits — reduces 8B model from ~16 GB to ~4 GB |
| `bnb_4bit_quant_type` | `"nf4"` | NormalFloat4 — optimal for normally-distributed (Gaussian) model weights |
| `bnb_4bit_use_double_quant` | `True` | Quantise the quantisation constants themselves; saves an extra ~0.5 GB |
| `bnb_4bit_compute_dtype` | `bfloat16` | Dequantise to BF16 for forward pass arithmetic; matches Llama's native dtype |

The tokeniser's `pad_token` is set to `eos_token` because Llama 3.1 has no dedicated padding token by default.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "microsoft/phi-2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Tokeniser
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"   # prevents warnings during SFT batching

# Base model in 4-bit
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False          # disable KV-cache; incompatible with gradient checkpointing
model.config.pretraining_tp = 1         # tensor parallelism = 1 for single-GPU Colab

print(f"Model loaded : {MODEL_ID}")
print(f"VRAM used    : {torch.cuda.memory_allocated() / 1e9:.2f} GB")

---
## Step 5 — Apply LoRA Configuration

### How LoRA Works

For each targeted weight matrix **W** (shape *d × k*), LoRA adds two low-rank matrices **A** (d × r) and **B** (r × k) where **r ≪ d**. The effective update is **ΔW = B · A**, scaled by **α/r**. During training, only **A** and **B** are updated; **W** stays frozen.

### Parameter Choices

| Parameter | Value | Reason |
|---|---|---|
| `r` | 16 | Rank — higher = more expressive but more params; 16 balances quality vs. size |
| `lora_alpha` | 32 | Scaling factor; effective learning rate scale = α/r = 2 |
| `target_modules` | `q_proj, v_proj` | Query and value projections — most impactful for instruction-following tasks; adding `k_proj`, `o_proj` would increase params but diminish returns |
| `lora_dropout` | 0.05 | Light regularisation; low because dataset is small |
| `bias` | `none` | Do not train bias terms — standard for LoRA |

`prepare_model_for_kbit_training` casts layer norms to float32 and enables gradient checkpointing hooks required for backpropagation through 4-bit quantised layers.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare for k-bit training (casts norms to float32, enables gradient checkpointing)
model = prepare_model_for_kbit_training(model)

# LoRA configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Inject LoRA adapters into the frozen base model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Required for gradient flow through frozen layers when using PEFT
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

---
## Step 6 — Load and Format the Training Dataset

The JSONL file has this schema per line:
```json
{"prompt": "Question: ...\nContext: ...\nAnswer:", "completion": " The answer text."}
```

SFTTrainer expects a single `text` column containing the **complete** training sequence (prompt + completion concatenated). We map each record to this format.

The full document is included as context (~400–1 000 tokens per example) because the model must learn to ground its answers in retrieved passages — matching the RAG inference pattern.

In [ ]:
from datasets import load_dataset

# Load raw JSONL
raw_ds = load_dataset("json", data_files=DRIVE_JSONL, split="train")
print(f"Raw examples : {len(raw_ds)}")
print(f"Columns      : {raw_ds.column_names}")

def format_example(example):
    # Concatenate prompt and completion into a single training text.
    # The model learns to predict the completion given the prompt prefix.
    return {"text": example["prompt"] + example["completion"]}

dataset = raw_ds.map(format_example, remove_columns=raw_ds.column_names)
print(f"Formatted examples : {len(dataset)}")
print(f"Columns            : {dataset.column_names}")

# Preview a single training example
sample = dataset[0]["text"]
print(f"\nSample (first 400 chars):\n{sample[:400]}")
print(f"...\nSample length: {len(sample.split())} words")

---
## Step 7 — Train with SFTTrainer

### Training Configuration

| Hyperparameter | Value | Reason |
|---|---|---|
| `num_train_epochs` | 3 | Small dataset (225 examples) — 3 epochs prevents underfitting without severe overfitting |
| `per_device_train_batch_size` | 4 | T4 has 16 GB; 4-bit model + gradient checkpointing fits batch of 4 |
| `gradient_accumulation_steps` | 4 | Effective batch = 4 × 4 = 16; stabilises gradient estimates |
| `learning_rate` | 2e-4 | Standard for LoRA fine-tuning; higher than full FT because only adapters train |
| `lr_scheduler_type` | cosine | Smooth decay; avoids sharp drops at epoch boundaries |
| `warmup_ratio` | 0.05 | 5% warmup steps protect against early instability |
| `optim` | `paged_adamw_8bit` | Quantised optimiser states — saves ~2 GB VRAM vs standard AdamW |
| `max_seq_length` | 2048 | Covers longest doc+question+answer sequences; examples are truncated if longer |
| `bf16` | `True` | BF16 arithmetic on T4/A100; more stable than FP16 for LLMs |

**Expected training time on T4**: ~20–35 minutes for 3 epochs over 225 examples.

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    # Output
    output_dir="./results",

    # Core hyperparameters
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,

    # Memory optimisation
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    fp16=False,
    bf16=True,

    # Sequence handling
    max_seq_length=2048,
    dataset_text_field="text",
    packing=False,          # keep context boundaries intact between examples

    # Logging
    logging_steps=10,
    logging_dir="./logs",
    save_strategy="epoch",
    report_to="none",       # set to "wandb" if you want loss curves tracked
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
)

print("Starting training...")
print(f"Steps per epoch : {len(dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}")
print(f"Total steps     : {trainer.args.max_steps if trainer.args.max_steps > 0 else 'auto'}")

train_result = trainer.train()

print(f"\nTraining complete.")
print(f"Runtime          : {train_result.metrics.get('train_runtime', 0):.0f}s")
print(f"Samples/second   : {train_result.metrics.get('train_samples_per_second', 0):.2f}")
print(f"Final train loss : {train_result.metrics.get('train_loss', 0):.4f}")

---
## Step 8 — Save Fine-tuned Adapter to Google Drive

Only the **LoRA adapter weights** are saved (~70 MB), not the full 4-bit model (~4 GB). At inference time, the adapter is loaded on top of the base model using `PeftModel.from_pretrained`.

**Saved files**:
- `adapter_config.json` — LoRA hyperparameters (r, alpha, target modules)
- `adapter_model.safetensors` — trained A and B matrices for q_proj and v_proj
- `tokenizer.json` + `tokenizer_config.json` — tokeniser saved alongside for reproducibility

In [ ]:
import os

# Save adapter weights (not the full model)
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"Adapter saved to: {ADAPTER_DIR}")
print("\nSaved files:")
for fname in sorted(os.listdir(ADAPTER_DIR)):
    fpath = os.path.join(ADAPTER_DIR, fname)
    size_mb = os.path.getsize(fpath) / 1e6
    print(f"  {fname:<45} {size_mb:>7.1f} MB")

total_mb = sum(
    os.path.getsize(os.path.join(ADAPTER_DIR, f)) / 1e6
    for f in os.listdir(ADAPTER_DIR)
)
print(f"\nTotal size: {total_mb:.1f} MB")

---
## Step 9 — Inference Test

Run three representative cybersecurity questions through the fine-tuned model to verify the adapter is working correctly before connecting it to the RAG pipeline evaluation dashboard.

The prompt format matches the training format exactly:
```
Question: {question}
Context: {retrieved_passage}
Answer:
```

In production, the `context` field is populated by the RAG retriever. Here we supply short context snippets to test response quality.

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# ── Helper: reload adapter from Drive for a clean inference session ────────
def load_finetuned_model(base_model_id: str, adapter_path: str):
    """Load base model + LoRA adapter for inference."""
    bnb_inf = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    tok = AutoTokenizer.from_pretrained(adapter_path)
    tok.pad_token = tok.eos_token
    base = AutoModelForCausalLM.from_pretrained(
        base_model_id,
        quantization_config=bnb_inf,
        device_map="auto",
    )
    peft_model = PeftModel.from_pretrained(base, adapter_path)
    peft_model.eval()
    return peft_model, tok


def ask(model, tokenizer, question: str, context: str, max_new_tokens: int = 200) -> str:
    """Run a single QA inference using the training prompt format."""
    prompt = f"Question: {question}\nContext: {context}\nAnswer:"
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1800,
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.2,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens (exclude the prompt)
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# ── Option A: use the trainer model (already in memory from training) ──────
# inf_model = trainer.model
# inf_model.eval()
# inf_tok = tokenizer

# ── Option B: reload from Drive (use this if running this cell in a new session)
print("Loading fine-tuned model from Drive...")
inf_model, inf_tok = load_finetuned_model(MODEL_ID, ADAPTER_DIR)
print("Model ready.\n")

# ── Three test questions ───────────────────────────────────────────────────
test_cases = [
    {
        "question": "What is SQL injection and how can it be prevented?",
        "context": (
            "SQL injection is a code injection attack where malicious SQL statements are inserted "
            "into input fields to manipulate the database. Attackers can bypass authentication, "
            "exfiltrate data, or drop tables. Prevention relies on parameterised queries and "
            "prepared statements, which separate SQL logic from user-supplied data. ORM frameworks "
            "provide built-in parameterisation. Input validation and least-privilege database "
            "accounts limit the blast radius of any successful injection."
        ),
    },
    {
        "question": "How does ransomware use encryption, and what is the most effective defence?",
        "context": (
            "Ransomware encrypts victim files using AES-256 symmetric encryption; the symmetric key "
            "is itself encrypted with the attacker's RSA-2048 public key so only they can recover it. "
            "Modern ransomware first exfiltrates data before encrypting (double extortion). "
            "Offline backups following the 3-2-1 rule (three copies, two media types, one offsite) "
            "are the most effective recovery mechanism. Patching, network segmentation, and EDR "
            "tools with behavioural analysis are key preventive controls."
        ),
    },
    {
        "question": "Why is multi-factor authentication more effective than passwords alone?",
        "context": (
            "Multi-factor authentication (MFA) requires two or more verification factors: something "
            "you know (password), something you have (TOTP token or hardware key), or something you "
            "are (biometric). The 2022 Verizon DBIR found that over 80% of hacking-related breaches "
            "involved stolen or brute-forced credentials. MFA ensures that a compromised password "
            "alone is insufficient for account takeover — the attacker must also compromise the "
            "second factor, which is significantly harder. FIDO2/WebAuthn hardware keys are "
            "phishing-resistant and represent the strongest MFA form."
        ),
    },
]

print("=" * 70)
for i, tc in enumerate(test_cases, 1):
    print(f"\nTest {i}/{len(test_cases)}: {tc['question']}")
    print("-" * 70)
    response = ask(inf_model, inf_tok, tc["question"], tc["context"])
    print(f"Answer: {response}")

print(f"\n{'=' * 70}")
print("Inference test complete. Adapter is working correctly.")

---
## Next Steps

The fine-tuned LoRA adapter saved at `ADAPTER_DIR` can now be integrated into the RAG pipeline evaluation dashboard:

1. **Download** `lora_adapter/` from Google Drive to `models/lora_adapter/` in the project repo
2. **Update `src/rag_pipeline.py`** to load the fine-tuned model:
   ```python
   from peft import PeftModel
   base = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, ...)
   finetuned_model = PeftModel.from_pretrained(base, "models/lora_adapter")
   ```
3. **Enable the fine-tuned column** in `src/app.py` evaluation dashboard tab (currently shows placeholder)
4. **Evaluate** on held-out questions using ROUGE-L, BERTScore, or faithfulness metrics

### Adapter Details

| Property | Value |
|---|---|
| Base model | `meta-llama/Llama-3.1-8B` |
| LoRA rank | 16 |
| LoRA alpha | 32 |
| Target modules | `q_proj`, `v_proj` |
| Training examples | 225 (45 docs × 5 pairs) |
| Epochs | 3 |
| Learning rate | 2e-4 (cosine decay) |